# Creazione di un Layer Convoluzionale 2D


In [ ]:
import tensorflow as tf

tf.keras.layers.Conv2D(
    filters = 32, # Numero di canali di output C2
    kernel_size=(3,3), # Dimensione spaziale del kernel 2x2
    strides=(1,1),
    padding="valid", # "valid" = no padding; "same" = padding con zeri per mantenere la dimensione
    data_format=None,
    dilation_rate=(1,1), # parametro per dilated convolution
    groups=1, # parametro per Grouping / Depthwise(G)
    activation=None,
    use_bias=True, # Aggiunta del termine di bias
    kernel_initializer="glorot_uniform",
    bias_initializer="zeros",
    # ... argomenti di regolarizzazione e constraint    
)


# Costruzione di una rete sequenziale di base (CNN)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Flatten, Dense

model = Sequential()

# --- Sottorete convoluzionale (Estrazione features) ---
model.add(Conv2D(32, (3,3), activation="relu", input_shape=(32,32,3)))
model.add(Conv2D(64, (3,3), activation="relu"))
model.add(Conv2D(64, (3,3), activation="relu"))

# Il layer Flatten prende il Tensore WxHxC2 di output della rete convoluzionale e lo srotola in un vettore 1D di dimensione (W*H*C)
model.add(Flatten())

# --- Sottorete Feed-Forward (Classificazione basata sulle feature) ---

model.add(Dense(64, activation="relu"))
model.add(Dense(10)) # Esempio di output per classificazione a 10 classi


# Implementazione layer di Pooling

In [ ]:
# Layer di MaxPooling

import tensorflow as tf

tf.keras.layers.MaxPooling2D(
    pool_size=(2,2),
    strides=None,
    padding="valid",
    data_format=None

)

# Layer di AvgPooling

tf.keras.layers.AveragePooling2D(
    pool_size=(2,2), # Dimensione della finestra per la media
    strides=None,
    padding="valid",
    data_format=None
)

## Implementazione Avanzata: Unpooling e Transposed Convolution

Per operazioni come la segmentazione semantica, dove è necessario riportare una feature map a una risoluzione maggiore, TensorFlow mette a disposizione layer specifici.

**Unpooling**

L'operazione di max unpooling non è presente nativamente nel core di base di TensorFlow, ma storicamente fa parte del pacchetto `tensorflow_addons` (sebbene sia considerata una libreria in via di deprecazione, resta il riferimento didattico e tecnico per questa operazione). Il layer richiede due input fondamentali: la feature map da ampliare e la matrice degli indici (le posizioni esatte in cui il max pooling aveva estratto il valore massimo).

Ecco un esempio pratico di implementazione:

In [1]:
import numpy as np
import tensorflow as tf
import tensorflow_addons as tfa


# Definizione del layer di input
x = tf.keras.Input(shape=(6,6,1))

# Applicazione del MaxPooling con salvataggio indici (argmax)
p, indices = tf.nn.max_pool_with_argmax(x,2,2, padding="SAME")

# Applicazione del MaxUnpooling utilizzando i valori e gli indici salvati
up = tfa.layers.MaxUnpooling2D()(p,indices)

# Creazione del modello per testare l'output
model = tf.keras.Model(inputs=x, outputs=[p,up])
model.summary()

# --- Valutazione pratica del modello ---
# Creiamo una matrice di test con un pattern specifico per vedere l'effetto

ones_matrix = np.ones((1,6,6,1))
ones_matrix[0,0,0,0] = 3
ones_matrix[0,-1,-1,0] = 5
res = model(ones_matrix)



print("MaxPooling output:")
print(res[0][0,:,:,0].numpy())
"""
Questo è un indice multiplo su un tensore 4D di forma (batch, altezza, larghezza, canali):

res[0] = primo output, cioè il pooling, di forma (1, 3, 3, 1)
[0, ...] = prendi la prima (e unica) immagine del batch
[:, :] = prendi tutte le righe e tutte le colonne
[0] = prendi il primo (e unico) canale
"""

print("MaxUnPooling2D output:")
print(res[1][0,:,:,0].numpy())

"""
Questo è un indice multiplo su un tensore 4D di forma (batch, altezza, larghezza, canali):

res[1] = secondo output, cioè Unpooling, di forma 6x6

"""


ModuleNotFoundError: No module named 'tensorflow_addons'